<a href="https://colab.research.google.com/github/AmplMrrr/compling-HW/blob/main/fine_tuning_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [1]:
# 0. Устанавливаю все, что нужно
!pip install transformers datasets evaluate accelerate gradio -q
!pip install huggingface_hub -q

import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
GPU доступен: True
Тип GPU: Tesla T4


In [2]:
# 0. Продолжаю подготовку
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# 1. Загрузка датасета
dataset = load_dataset("ag_news")
print(f"Датасет загружен. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Датасет загружен. Train: 120000, Test: 7600


In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [5]:
dataset['test'][0]

{'text': "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.",
 'label': 2}

In [6]:
# 2. Загрузка модели и токенизатора
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4  # 0 = World, 1 = Sports, 2 = Business, 3 = Sci/Tech
).to(device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# 3. Подготовка данных (токенизация)
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(5000))

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [8]:
# 4. Настройка обучения
training_args = TrainingArguments(
    output_dir="./results-imdb",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    logging_steps=500,
)

In [9]:
# 5.1 Настройка метрик
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [10]:
# 5.2 Обучение
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.196127,0.181668,0.941000
2,0.134368,0.190673,0.946200
3,0.089297,0.226762,0.944200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=22500, training_loss=0.15308388977050782, metrics={'train_runtime': 3275.6891, 'train_samples_per_second': 109.901, 'train_steps_per_second': 6.869, 'total_flos': 8558812764910464.0, 'train_loss': 0.15308388977050782, 'epoch': 3.0})

In [11]:
# 6. Выведение accuracy на тестовой выборке
eval_results = trainer.evaluate()
print(f"\nEvaluation results: {eval_results}")


Evaluation results: {'eval_loss': 0.19068020582199097, 'eval_accuracy': 0.9464, 'eval_runtime': 15.0517, 'eval_samples_per_second': 332.189, 'eval_steps_per_second': 20.795, 'epoch': 3.0}


In [12]:
# 7. Сохранение модели
model.save_pretrained("./ag_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
# 8. Тест на новых примерах. Я надеюсь, вы хотя улыбнетесь от них ;)
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

test_texts = [
    "Russian athletes got to the Olympics through the VPN.",
    "The girl sold cucumbers clandestinely and was able to pay off the mortgage.",
    "Famous developer Elon Musk mentioned the name of his new child, but our respondent did not remember it."
]

for text in test_texts:
    result = classifier(text)[0]
    print(f"Text: {text}\nSentiment: {result['label']}, Score: {result['score']:.4f}\n")

Text: Russian athletes got to the Olympics through the VPN.
Sentiment: LABEL_0, Score: 0.5092

Text: The girl sold cucumbers clandestinely and was able to pay off the mortgage.
Sentiment: LABEL_2, Score: 0.9624

Text: Famous developer Elon Musk mentioned the name of his new child, but our respondent did not remember it.
Sentiment: LABEL_2, Score: 0.9857



Мини итоги:
- Модель верно определила только один из трех придуманных мною новостей (вторую новость, которая действительно про бизнес).
- Интересно предсказание для первой выдуманной новости, поскольку в предложении встречалось два слова, отсылающие к теме спорта ("спортсмены" и "олимпиада"), и два слова, которые можно было бы отнести к теме "мир" ("русские" и "ВПН"). Несмотря на то, что модель указала тему "мир", ее уверенность в этом решении равна примерно 50%, что соотносится с приведенным выше анализом. Можно предположить, что модель действительно выбирала между темами "мир" и "спорт".
- Видимо, Илон Макс, в первую очередь, именно самый богатый человек на планете, а не какой-то там "разработчик".